# IF3270 Tugas Besar 2 — Kelompok 61
## CNN & Image Captioning with RNN / LSTM

| | |
|---|---|
| **Mata kuliah** | IF3270 Machine Learning |
| **Dataset CNN** | Intel Image Classification (6 kelas) |
| **Dataset Caption** | Flickr8k |
| **Framework** | TensorFlow / Keras + NumPy (from-scratch) |

## Daftar Isi

- [0. Setup & Import](#0.-Setup-&-Import)
- [**A. CNN**](#A.-CNN)
  - [A.1. Perbandingan Shared vs Non-Shared Parameter](#A.1.)
  - [A.2. Pengaruh Jumlah Layer Konvolusi](#A.2.)
  - [A.3. Pengaruh Banyak Filter per Layer](#A.3.)
  - [A.4. Pengaruh Ukuran Filter](#A.4.)
  - [A.5. Pengaruh Jenis Pooling Layer](#A.5.)
  - [A.6. Perbandingan Keras vs From Scratch (CNN)](#A.6.)
- [**B. RNN & LSTM — Image Captioning**](#B.-RNN-&-LSTM)
  - [B.1. Perbandingan Jumlah Layer: RNN](#B.1.)
  - [B.2. Perbandingan Jumlah Layer: LSTM](#B.2.)
  - [B.3. Perbandingan Ukuran Hidden State: RNN](#B.3.)
  - [B.4. Perbandingan Ukuran Hidden State: LSTM](#B.4.)
  - [B.5. Perbandingan RNN vs LSTM](#B.5.)
  - [B.6. Perbandingan Keras vs From Scratch (Captioning)](#B.6.)
  - [B.7. Pengaruh Panjang Maksimum Caption terhadap BLEU-4](#B.7.)

### 0. Setup & Import

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import sys, os, json, time, warnings
from pathlib import Path
from collections import defaultdict, Counter

# ── Third-party ────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

warnings.filterwarnings('ignore')

# ── Path detection ─────────────────────────────────────────────────────────────
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()
    for _ in range(6):
        if (ROOT / 'src').exists() and (ROOT / 'requirements.txt').exists():
            break
        ROOT = ROOT.parent

SRC_DIR      = ROOT / 'src'
NOTEBOOK_DIR = SRC_DIR / 'notebook'
TRAINING_DIR = NOTEBOOK_DIR / 'training'
EVAL_DIR     = NOTEBOOK_DIR / 'evaluation'

for p in [str(SRC_DIR), str(NOTEBOOK_DIR), str(TRAINING_DIR), str(EVAL_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Project paths ──────────────────────────────────────────────────────────────
MODELS_CNN_DIR = ROOT / 'models'
MODELS_CAP_DIR = ROOT / 'models_captioning'
PROC_DIR       = ROOT / 'data_processed'
DATASET_DIR    = ROOT / 'dataset'

# Intel dataset: support both download-script layout and flat layout
INTEL_DIR = next(
    (p for p in [
        DATASET_DIR / 'intel-image-classification',
        DATASET_DIR,
    ] if (p / 'seg_test').exists()),
    DATASET_DIR / 'intel-image-classification',
)
CNN_TEST_DIR  = INTEL_DIR / 'seg_test'  / 'seg_test'
CNN_TRAIN_DIR = INTEL_DIR / 'seg_train' / 'seg_train'
FLICKR_IMG_DIR = DATASET_DIR / 'flickr8k' / 'Images'

# ── Global constants ───────────────────────────────────────────────────────────
IMG_SIZE      = (150, 150)
N_CLASSES     = 6
CLASS_NAMES   = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
BATCH_SIZE    = 32
EMBED_DIM     = 256
MAX_LEN       = 35
SCRATCH_BATCH = 16   # smaller batch for NumPy (slower than Keras)

print(f'ROOT             : {ROOT}')
print(f'CNN models dir   : {MODELS_CNN_DIR}')
print(f'Caption models   : {MODELS_CAP_DIR}')
print(f'CNN test dir     : {CNN_TEST_DIR}')

In [ ]:
# ── Dataset verification ──────────────────────────────────────────────────────
FLICKR_DIR = DATASET_DIR / 'flickr8k'

missing = []
if not CNN_TEST_DIR.exists():
    missing.append(f'Intel seg_test  -> {CNN_TEST_DIR}')
if not FLICKR_DIR.exists():
    missing.append(f'Flickr8k        -> {FLICKR_DIR}')

if missing:
    print('=' * 70)
    print('⚠  DATASET BELUM ADA — jalankan dulu:')
    print('   python scripts/download_datasets.py')
    print()
    for m in missing:
        print(f'   MISSING: {m}')
    print('=' * 70)
else:
    n_test  = sum(1 for _ in CNN_TEST_DIR.glob('*/*.jpg'))  if CNN_TEST_DIR.exists()  else 0
    n_train = sum(1 for _ in CNN_TRAIN_DIR.glob('*/*.jpg')) if CNN_TRAIN_DIR.exists() else 0
    n_flick = sum(1 for _ in FLICKR_IMG_DIR.glob('*.jpg')) if FLICKR_IMG_DIR.exists() else 0
    print('Dataset OK:')
    print(f'  Intel train : {n_train:,} gambar')
    print(f'  Intel test  : {n_test:,} gambar')
    print(f'  Flickr8k    : {n_flick:,} gambar')

In [ ]:
# ── Library versions & GPU check ──────────────────────────────────────────────
print(f'NumPy      : {np.__version__}')

try:
    import tensorflow as tf
    from tensorflow import keras
    print(f'TensorFlow : {tf.__version__}')
    gpus = tf.config.list_physical_devices('GPU')
    print(f'GPU        : {gpus if gpus else "none (CPU only)"}')
    _HAS_TF = True
except ImportError:
    print('⚠  TensorFlow tidak terinstall — section Keras akan diskip.')
    _HAS_TF = False

try:
    from sklearn.metrics import f1_score, confusion_matrix, classification_report
    import sklearn
    print(f'scikit-learn : {sklearn.__version__}')
    _HAS_SKLEARN = True
except ImportError:
    print('⚠  scikit-learn tidak terinstall.')
    _HAS_SKLEARN = False

try:
    import nltk
    from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction
    from nltk.translate.meteor_score import meteor_score
    print(f'NLTK       : {nltk.__version__}')
    _HAS_NLTK = True
except ImportError:
    print('⚠  NLTK tidak terinstall — BLEU/METEOR tidak tersedia.')
    _HAS_NLTK = False

In [ ]:
# ── Helper: load CNN test images ──────────────────────────────────────────────
def load_cnn_test_data(test_dir=None):
    if test_dir is None:
        test_dir = CNN_TEST_DIR
    images, labels = [], []
    for idx, cls in enumerate(CLASS_NAMES):
        cls_dir = Path(test_dir) / cls
        if not cls_dir.exists():
            continue
        for p in sorted(cls_dir.glob('*.jpg')):
            img = Image.open(p).convert('RGB').resize((IMG_SIZE[1], IMG_SIZE[0]))
            images.append(np.array(img, dtype=np.float32) / 255.0)
            labels.append(idx)
    if not images:
        return np.empty((0, *IMG_SIZE, 3), dtype=np.float32), np.empty(0, dtype=np.int32)
    return np.stack(images), np.array(labels, dtype=np.int32)


# ── Helper: Keras inference ────────────────────────────────────────────────────
def predict_keras(model, x, batch_size=32):
    return np.argmax(model.predict(x, batch_size=batch_size, verbose=0), axis=1)


# ── Helper: from-scratch Diffable graph inference ─────────────────────────────
def predict_scratch(input_node, output_node, x, batch_size=SCRATCH_BATCH):
    preds = []
    for s in range(0, len(x), batch_size):
        batch = x[s:s + batch_size]
        input_node.clear_values()
        input_node._value = batch
        out = output_node.get_value()
        preds.append(np.argmax(out, axis=1))
    return np.concatenate(preds)


# ── Helper: loss curve plot ────────────────────────────────────────────────────
def plot_loss_curves(histories, labels, title='Loss Curve', figsize=(12, 5), ax=None):
    own = ax is None
    if own:
        _, ax = plt.subplots(figsize=figsize)
    colors = plt.cm.tab10.colors
    for i, (h, lbl) in enumerate(zip(histories, labels)):
        c = colors[i % 10]
        ep = range(1, len(h['train_loss']) + 1)
        ax.plot(ep, h['train_loss'], '-',  color=c, label=f'{lbl} train')
        ax.plot(ep, h['val_loss'],   '--', color=c, label=f'{lbl} val', alpha=0.7)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss',  fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9, ncol=max(1, len(histories)//3))
    ax.grid(True, alpha=0.3)
    if own:
        plt.tight_layout()
        plt.show()


# ── Helper: bar chart ─────────────────────────────────────────────────────────
def compare_bar(values, labels, title='', ylabel='', figsize=(10, 5),
                color='steelblue', highlight_best=True, ax=None):
    own = ax is None
    if own:
        _, ax = plt.subplots(figsize=figsize)
    colors = [color] * len(values) if isinstance(color, str) else list(color)
    if highlight_best and values:
        colors[int(np.argmax(values))] = 'tomato'
    bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=0.8)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(values) * 0.01,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel(ylabel or '', fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, max(values) * 1.18 if values else 1)
    if own:
        plt.tight_layout()
        plt.show()


# ── Helper: multi-panel loss grid ─────────────────────────────────────────────
def plot_loss_grid(results_subset, n_cols=3, figsize_per=(4, 3)):
    n = len(results_subset)
    if n == 0:
        return
    n_rows = (n + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(figsize_per[0]*n_cols, figsize_per[1]*n_rows))
    axes = np.array(axes).flatten()
    for i, r in enumerate(results_subset):
        ax = axes[i]
        ep = range(1, len(r['train_loss']) + 1)
        ax.plot(ep, r['train_loss'], label='train')
        ax.plot(ep, r['val_loss'],   '--', label='val', alpha=0.8)
        ax.set_title(r['tag'], fontsize=10, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=7)
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    plt.tight_layout()
    plt.show()


# ── Helper: print ASCII table ─────────────────────────────────────────────────
def print_table(headers, rows):
    widths = [max(len(str(h)), max((len(str(r[i])) for r in rows), default=0))
              for i, h in enumerate(headers)]
    fmt = '  '.join(f'{{:<{w}}}' for w in widths)
    print(fmt.format(*headers))
    print('  '.join('-' * w for w in widths))
    for row in rows:
        print(fmt.format(*[str(v) for v in row]))


print('Helper functions defined ✓')

---
### A. CNN — Intel Image Classification

#### A.1. Perbandingan Shared vs Non-Shared Parameter

Membandingkan **Conv2D** (kernel bersama / *shared weights*) dengan
**LocallyConnected2D** (kernel per-posisi / *non-shared*) menggunakan
arsitektur CNN terbaik hasil pelatihan.

- **Shared**: satu set kernel digunakan di semua posisi spasial
- **Non-shared**: setiap posisi output memiliki kernel tersendiri (lebih ekspresif, lebih besar)

In [ ]:
# ── A.1 Load CNN training results ─────────────────────────────────────────────
CNN_RESULTS_FILE = MODELS_CNN_DIR / 'training_results.json'

if not CNN_RESULTS_FILE.exists():
    print(f'⚠  {CNN_RESULTS_FILE} belum ada.')
    print('   Jalankan: python src/notebook/cnn_training.py')
    _cnn_results = []
    _best_cnn    = None
else:
    with open(CNN_RESULTS_FILE) as f:
        _cnn_results = json.load(f)
    _best_cnn = max(_cnn_results, key=lambda r: r['test_macro_f1'])
    print(f'Loaded {len(_cnn_results)} CNN experiment results.')
    print(f'Best model : {_best_cnn["tag"]}')
    print(f'Test F1    : {_best_cnn["test_macro_f1"]:.4f}')
    print(f'Parameters : {_best_cnn["total_params"]:,}')

In [ ]:
# ── A.1 Build from-scratch graphs ─────────────────────────────────────────────
if not _HAS_TF or _best_cnn is None:
    print('⚠  Skip — TF atau model belum tersedia.')
else:
    from algorithm.core.parameter import Parameter, Input
    from algorithm.function.relu   import ReLU
    from algorithm.function.softmax import Softmax
    from algorithm.neural.conv      import Conv2D
    from algorithm.neural.local     import LocallyConnected2D
    from algorithm.neural.maxpool   import MaxPool2D
    from algorithm.neural.avgpool   import AvgPool2D
    from algorithm.neural.reshape   import Flatten
    from algorithm.neural.linear    import Linear
    from cnn_evaluation import build_scratch_model, load_test_data

    _keras_a1 = keras.models.load_model(_best_cnn['saved_to'])
    print(f'Keras model loaded: {_best_cnn["tag"]}')

    _inp_shared, _out_shared = build_scratch_model(_keras_a1, use_locally_connected=False)
    _inp_lc,     _out_lc     = build_scratch_model(_keras_a1, use_locally_connected=True)
    print('From-scratch graphs built (Conv2D & LocallyConnected2D) ✓')

In [ ]:
# ── A.1 Run inference and compute metrics ─────────────────────────────────────
if not _HAS_TF or _best_cnn is None:
    print('⚠  Skip.')
else:
    print('Loading CNN test data ...')
    _x_cnn, _y_cnn = load_test_data(CNN_TEST_DIR)
    print(f'  {len(_x_cnn)} test images')

    # Keras
    t0 = time.time()
    _kpreds_a1  = predict_keras(_keras_a1, _x_cnn)
    _ktime_a1   = time.time() - t0
    _kf1_a1     = f1_score(_y_cnn, _kpreds_a1, average='macro')

    # Scratch Conv2D (shared)
    t0 = time.time()
    _sp_shared  = predict_scratch(_inp_shared, _out_shared, _x_cnn)
    _st_shared  = time.time() - t0
    _sf1_shared = f1_score(_y_cnn, _sp_shared, average='macro')

    # Scratch LocallyConnected2D (non-shared)
    t0 = time.time()
    _sp_lc      = predict_scratch(_inp_lc, _out_lc, _x_cnn)
    _st_lc      = time.time() - t0
    _sf1_lc     = f1_score(_y_cnn, _sp_lc, average='macro')

    print('=' * 70)
    print('A.1  PERBANDINGAN SHARED VS NON-SHARED PARAMETER')
    print('=' * 70)
    print(f'  Model (terbaik)        : {_best_cnn["tag"]}')
    print(f'  Jumlah parameter       : {_best_cnn["total_params"]:,}')
    print()
    print(f'  Keras (baseline)  F1   : {_kf1_a1:.4f}   ({_ktime_a1:.1f}s)')
    print(f'  Scratch Conv2D    F1   : {_sf1_shared:.4f}   ({_st_shared:.1f}s)')
    print(f'  Scratch LC2D      F1   : {_sf1_lc:.4f}   ({_st_lc:.1f}s)')
    print()
    print(f'  |diff| Keras - Conv2D  : {abs(_kf1_a1 - _sf1_shared):.6f}')
    print(f'  |diff| Keras - LC2D    : {abs(_kf1_a1 - _sf1_lc):.6f}')

In [ ]:
# ── A.1 Plots ─────────────────────────────────────────────────────────────────
if not _HAS_TF or _best_cnn is None:
    print('⚠  Skip.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    _a1_labels = ['Keras\n(baseline)', 'Scratch\nConv2D\n(shared)', 'Scratch\nLC2D\n(non-shared)']
    _a1_f1s    = [_kf1_a1, _sf1_shared, _sf1_lc]
    _a1_times  = [_ktime_a1, _st_shared, _st_lc]
    _a1_colors = ['steelblue', 'mediumseagreen', 'darkorange']

    bars = axes[0].bar(_a1_labels, _a1_f1s, color=_a1_colors, edgecolor='white')
    for bar, v in zip(bars, _a1_f1s):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                     f'{v:.4f}', ha='center', va='bottom', fontsize=10)
    axes[0].set_title('Macro F1 Score', fontsize=13, fontweight='bold')
    axes[0].set_ylabel('Macro F1', fontsize=12)
    axes[0].set_ylim(0, 1.1)
    axes[0].grid(True, alpha=0.3, axis='y')

    bars2 = axes[1].bar(_a1_labels, _a1_times, color=_a1_colors, edgecolor='white')
    for bar, v in zip(bars2, _a1_times):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                     f'{v:.1f}s', ha='center', va='bottom', fontsize=10)
    axes[1].set_title('Waktu Inferensi (detik)', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('Waktu (s)', fontsize=12)
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.suptitle(f'A.1 Shared vs Non-Shared — {_best_cnn["tag"]}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Loss curve
    fig2, ax2 = plt.subplots(figsize=(10, 4))
    plot_loss_curves(
        [{'train_loss': _best_cnn['train_loss'], 'val_loss': _best_cnn['val_loss']}],
        [_best_cnn['tag']],
        title=f'A.1 Loss Curve — {_best_cnn["tag"]}',
        ax=ax2,
    )
    plt.tight_layout()
    plt.show()

#### [Kesimpulan A.1]

> *Isi setelah eksperimen selesai dijalankan.*

#### A.2. Pengaruh Jumlah Layer Konvolusi

Membandingkan model dengan **2 block** vs **3 block** konvolusi, dengan
`filters_key`, `kernel_size`, dan `pool_type` dikontrol pada nilai yang paling umum.

In [ ]:
# ── A.2 Filter results by n_blocks ────────────────────────────────────────────
if not _cnn_results:
    print('⚠  Skip — training_results.json belum ada.')
    _a2_results = []
else:
    # Fix other factors at their most common value
    _fix_fkey  = Counter(r['config']['filters_key'] for r in _cnn_results).most_common(1)[0][0]
    _fix_ksize = Counter(tuple(r['config']['kernel_size']) for r in _cnn_results).most_common(1)[0][0]
    _fix_pool  = Counter(r['config']['pool_type'] for r in _cnn_results).most_common(1)[0][0]

    _a2_results = sorted(
        [r for r in _cnn_results
         if r['config']['filters_key'] == _fix_fkey
         and tuple(r['config']['kernel_size']) == _fix_ksize
         and r['config']['pool_type'] == _fix_pool],
        key=lambda r: r['config']['n_blocks'],
    )
    print(f'Fixed: filters_key={_fix_fkey}, kernel={_fix_ksize}, pool={_fix_pool}')
    print(f'Variasi n_blocks : {[r["config"]["n_blocks"] for r in _a2_results]}')

    print('=' * 70)
    print('A.2  PENGARUH JUMLAH LAYER KONVOLUSI')
    print('=' * 70)
    print_table(
        ['Tag', 'n_blocks', 'Filters', 'val F1', 'test F1', 'Params'],
        [(r['tag'], r['config']['n_blocks'], str(r['config']['filters']),
          f"{r['val_macro_f1']:.4f}", f"{r['test_macro_f1']:.4f}",
          f"{r['total_params']:,}") for r in _a2_results]
    )

In [ ]:
# ── A.2 Plots ─────────────────────────────────────────────────────────────────
if not _a2_results:
    print('⚠  Skip.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    compare_bar(
        [r['test_macro_f1'] for r in _a2_results],
        [f'n={r["config"]["n_blocks"]} blocks' for r in _a2_results],
        title='Test Macro F1 per Jumlah Block Konvolusi',
        ylabel='Macro F1', ax=axes[0],
    )
    plot_loss_curves(
        [{'train_loss': r['train_loss'], 'val_loss': r['val_loss']} for r in _a2_results],
        [r['tag'] for r in _a2_results],
        title='Loss Curves per Jumlah Block', ax=axes[1],
    )
    plt.suptitle('A.2 Pengaruh Jumlah Layer Konvolusi', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    plot_loss_grid(_a2_results, n_cols=len(_a2_results))

#### [Kesimpulan A.2]

> *Isi setelah eksperimen selesai dijalankan.*

#### A.3. Pengaruh Banyak Filter per Layer Konvolusi

Membandingkan konfigurasi **small** (32→64 / 32→64→128) vs **large** (64→128 / 64→128→256),
dengan jumlah block, kernel size, dan pool type dikontrol.

In [ ]:
# ── A.3 Filter results by filters_key ─────────────────────────────────────────
if not _cnn_results:
    print('⚠  Skip.')
    _a3_results = []
else:
    _a3_nblocks = Counter(r['config']['n_blocks'] for r in _cnn_results).most_common(1)[0][0]
    _a3_ksize   = Counter(tuple(r['config']['kernel_size']) for r in _cnn_results).most_common(1)[0][0]
    _a3_pool    = Counter(r['config']['pool_type'] for r in _cnn_results).most_common(1)[0][0]

    _a3_results = sorted(
        [r for r in _cnn_results
         if r['config']['n_blocks'] == _a3_nblocks
         and tuple(r['config']['kernel_size']) == _a3_ksize
         and r['config']['pool_type'] == _a3_pool],
        key=lambda r: r['config']['filters_key'],
    )
    print(f'Fixed: n_blocks={_a3_nblocks}, kernel={_a3_ksize}, pool={_a3_pool}')
    print(f'Variasi filters_key : {[r["config"]["filters_key"] for r in _a3_results]}')

    print('=' * 70)
    print('A.3  PENGARUH BANYAK FILTER PER LAYER KONVOLUSI')
    print('=' * 70)
    print_table(
        ['Tag', 'filters_key', 'Filters', 'val F1', 'test F1', 'Params'],
        [(r['tag'], r['config']['filters_key'], str(r['config']['filters']),
          f"{r['val_macro_f1']:.4f}", f"{r['test_macro_f1']:.4f}",
          f"{r['total_params']:,}") for r in _a3_results]
    )

In [ ]:
# ── A.3 Plots ─────────────────────────────────────────────────────────────────
if not _a3_results:
    print('⚠  Skip.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    compare_bar(
        [r['test_macro_f1'] for r in _a3_results],
        [r['config']['filters_key'] + '\n' + str(r['config']['filters']) for r in _a3_results],
        title='Test Macro F1 per Konfigurasi Filter',
        ylabel='Macro F1', ax=axes[0],
        color=['steelblue', 'darkorange'],
        highlight_best=False,
    )
    plot_loss_curves(
        [{'train_loss': r['train_loss'], 'val_loss': r['val_loss']} for r in _a3_results],
        [r['tag'] for r in _a3_results],
        title='Loss Curves per Konfigurasi Filter', ax=axes[1],
    )
    plt.suptitle('A.3 Pengaruh Banyak Filter per Layer', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    plot_loss_grid(_a3_results, n_cols=len(_a3_results))

#### [Kesimpulan A.3]

> *Isi setelah eksperimen selesai dijalankan.*

#### A.4. Pengaruh Ukuran Filter per Layer Konvolusi

Membandingkan kernel **3×3** vs **5×5**, dengan jumlah block, konfigurasi filter,
dan pool type dikontrol.

In [ ]:
# ── A.4 Filter results by kernel_size ─────────────────────────────────────────
if not _cnn_results:
    print('⚠  Skip.')
    _a4_results = []
else:
    _a4_nblocks = Counter(r['config']['n_blocks'] for r in _cnn_results).most_common(1)[0][0]
    _a4_fkey    = Counter(r['config']['filters_key'] for r in _cnn_results).most_common(1)[0][0]
    _a4_pool    = Counter(r['config']['pool_type'] for r in _cnn_results).most_common(1)[0][0]

    _a4_results = sorted(
        [r for r in _cnn_results
         if r['config']['n_blocks'] == _a4_nblocks
         and r['config']['filters_key'] == _a4_fkey
         and r['config']['pool_type'] == _a4_pool],
        key=lambda r: r['config']['kernel_size'][0],
    )
    print(f'Fixed: n_blocks={_a4_nblocks}, filters_key={_a4_fkey}, pool={_a4_pool}')
    print(f'Variasi kernel_size : {[r["config"]["kernel_size"] for r in _a4_results]}')

    print('=' * 70)
    print('A.4  PENGARUH UKURAN FILTER')
    print('=' * 70)
    print_table(
        ['Tag', 'kernel_size', 'val F1', 'test F1', 'Params'],
        [(r['tag'], f'{r["config"]["kernel_size"][0]}x{r["config"]["kernel_size"][1]}',
          f"{r['val_macro_f1']:.4f}", f"{r['test_macro_f1']:.4f}",
          f"{r['total_params']:,}") for r in _a4_results]
    )

In [ ]:
# ── A.4 Plots ─────────────────────────────────────────────────────────────────
if not _a4_results:
    print('⚠  Skip.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    compare_bar(
        [r['test_macro_f1'] for r in _a4_results],
        [f'{r["config"]["kernel_size"][0]}×{r["config"]["kernel_size"][1]}' for r in _a4_results],
        title='Test Macro F1 per Ukuran Kernel',
        ylabel='Macro F1', ax=axes[0],
        color=['steelblue', 'darkorange'], highlight_best=False,
    )
    plot_loss_curves(
        [{'train_loss': r['train_loss'], 'val_loss': r['val_loss']} for r in _a4_results],
        [r['tag'] for r in _a4_results],
        title='Loss Curves per Ukuran Kernel', ax=axes[1],
    )
    plt.suptitle('A.4 Pengaruh Ukuran Filter', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    plot_loss_grid(_a4_results, n_cols=len(_a4_results))

#### [Kesimpulan A.4]

> *Isi setelah eksperimen selesai dijalankan.*

#### A.5. Pengaruh Jenis Pooling Layer

Membandingkan **MaxPooling2D** vs **AveragePooling2D** dengan faktor lain dikontrol.

In [ ]:
# ── A.5 Filter results by pool_type ───────────────────────────────────────────
if not _cnn_results:
    print('⚠  Skip.')
    _a5_results = []
else:
    _a5_nblocks = Counter(r['config']['n_blocks'] for r in _cnn_results).most_common(1)[0][0]
    _a5_fkey    = Counter(r['config']['filters_key'] for r in _cnn_results).most_common(1)[0][0]
    _a5_ksize   = Counter(tuple(r['config']['kernel_size']) for r in _cnn_results).most_common(1)[0][0]

    _a5_results = sorted(
        [r for r in _cnn_results
         if r['config']['n_blocks'] == _a5_nblocks
         and r['config']['filters_key'] == _a5_fkey
         and tuple(r['config']['kernel_size']) == _a5_ksize],
        key=lambda r: r['config']['pool_type'],
    )
    print(f'Fixed: n_blocks={_a5_nblocks}, filters_key={_a5_fkey}, kernel={_a5_ksize}')
    print(f'Variasi pool_type : {[r["config"]["pool_type"] for r in _a5_results]}')

    print('=' * 70)
    print('A.5  PENGARUH JENIS POOLING LAYER')
    print('=' * 70)
    print_table(
        ['Tag', 'pool_type', 'val F1', 'test F1', 'Params'],
        [(r['tag'], r['config']['pool_type'],
          f"{r['val_macro_f1']:.4f}", f"{r['test_macro_f1']:.4f}",
          f"{r['total_params']:,}") for r in _a5_results]
    )

In [ ]:
# ── A.5 Plots ─────────────────────────────────────────────────────────────────
if not _a5_results:
    print('⚠  Skip.')
else:
    _pool_colors = {'max': 'steelblue', 'avg': 'darkorange'}
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    compare_bar(
        [r['test_macro_f1'] for r in _a5_results],
        [r['config']['pool_type'].capitalize() + 'Pool' for r in _a5_results],
        title='Test Macro F1 per Jenis Pooling',
        ylabel='Macro F1', ax=axes[0],
        color=[_pool_colors[r['config']['pool_type']] for r in _a5_results],
        highlight_best=False,
    )
    plot_loss_curves(
        [{'train_loss': r['train_loss'], 'val_loss': r['val_loss']} for r in _a5_results],
        [r['tag'] for r in _a5_results],
        title='Loss Curves per Jenis Pooling', ax=axes[1],
    )
    plt.suptitle('A.5 Pengaruh Jenis Pooling Layer', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    plot_loss_grid(_a5_results, n_cols=len(_a5_results))

#### [Kesimpulan A.5]

> *Isi setelah eksperimen selesai dijalankan.*

#### A.6. Perbandingan Keras vs From Scratch (CNN)

Evaluasi pada arsitektur terbaik: Keras (TF backend), from-scratch Conv2D (shared),
dan from-scratch LocallyConnected2D (non-shared).

In [ ]:
# ── A.6 Keras vs From Scratch ─────────────────────────────────────────────────
# Reuse A.1 variables if available; otherwise recompute
if not _HAS_TF or _best_cnn is None:
    print('⚠  Skip — TF atau model belum tersedia.')
else:
    try:
        _kf1_a6     = _kf1_a1
        _ktime_a6   = _ktime_a1
        _sf1_a6s    = _sf1_shared
        _st_a6s     = _st_shared
        _sf1_a6lc   = _sf1_lc
        _st_a6lc    = _st_lc
        _kpreds_a6  = _kpreds_a1
        _sp_a6s     = _sp_shared
        _sp_a6lc    = _sp_lc
        print('(Reusing A.1 results)')
    except NameError:
        print('Recomputing (A.1 not run) ...')
        from cnn_evaluation import build_scratch_model, load_test_data
        _keras_a1   = keras.models.load_model(_best_cnn['saved_to'])
        _x_cnn, _y_cnn = load_test_data(CNN_TEST_DIR)
        t0 = time.time(); _kpreds_a6 = predict_keras(_keras_a1, _x_cnn); _ktime_a6 = time.time()-t0
        _kf1_a6  = f1_score(_y_cnn, _kpreds_a6,  average='macro')
        _inp_s, _out_s = build_scratch_model(_keras_a1, False)
        _inp_l, _out_l = build_scratch_model(_keras_a1, True)
        t0 = time.time(); _sp_a6s  = predict_scratch(_inp_s, _out_s, _x_cnn); _st_a6s  = time.time()-t0
        t0 = time.time(); _sp_a6lc = predict_scratch(_inp_l, _out_l, _x_cnn); _st_a6lc = time.time()-t0
        _sf1_a6s  = f1_score(_y_cnn, _sp_a6s,  average='macro')
        _sf1_a6lc = f1_score(_y_cnn, _sp_a6lc, average='macro')

    print('=' * 70)
    print('A.6  PERBANDINGAN KERAS VS FROM SCRATCH (CNN)')
    print('=' * 70)
    print(f'  Arsitektur             : {_best_cnn["tag"]}')
    print(f'  Jumlah parameter       : {_best_cnn["total_params"]:,}')
    print()
    print(f'  Keras (baseline)  F1   : {_kf1_a6:.4f}   ({_ktime_a6:.1f}s)')
    print(f'  Scratch Conv2D    F1   : {_sf1_a6s:.4f}   ({_st_a6s:.1f}s)')
    print(f'  Scratch LC2D      F1   : {_sf1_a6lc:.4f}   ({_st_a6lc:.1f}s)')
    print()
    print(f'  |diff| Keras - Conv2D  : {abs(_kf1_a6 - _sf1_a6s):.6f}')
    print(f'  |diff| Keras - LC2D    : {abs(_kf1_a6 - _sf1_a6lc):.6f}')

In [ ]:
# ── A.6 Confusion matrices ────────────────────────────────────────────────────
if not _HAS_TF or _best_cnn is None:
    print('⚠  Skip.')
else:
    try:
        _preds_set = [_kpreds_a6, _sp_a6s, _sp_a6lc]
    except NameError:
        print('⚠  Run cell above first.')
        _preds_set = None

    if _preds_set is not None:
        _cm_titles = ['Keras\n(baseline)', 'Scratch Conv2D\n(shared)', 'Scratch LC2D\n(non-shared)']
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        short = [c[:3] for c in CLASS_NAMES]
        for ax, preds, title in zip(axes, _preds_set, _cm_titles):
            cm = confusion_matrix(_y_cnn, preds)
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                        xticklabels=short, yticklabels=short)
            ax.set_title(title, fontsize=12, fontweight='bold')
            ax.set_xlabel('Predicted', fontsize=10)
            ax.set_ylabel('True', fontsize=10)
        plt.suptitle(f'A.6 Confusion Matrix — {_best_cnn["tag"]}',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

#### [Kesimpulan A.6]

> *Isi setelah eksperimen selesai dijalankan.*

---
### B. RNN & LSTM — Image Captioning (Flickr8k)

In [ ]:
# ── B. Common setup: load captioning results ──────────────────────────────────
CAP_RESULTS_FILE = MODELS_CAP_DIR / 'captioning_results.json'
CAP_EVAL_FILE    = MODELS_CAP_DIR / 'evaluation_results.json'

if not CAP_RESULTS_FILE.exists():
    print(f'⚠  {CAP_RESULTS_FILE} belum ada.')
    print('   Jalankan: python src/notebook/captioning_training.py')
    _cap_results = []
else:
    with open(CAP_RESULTS_FILE) as f:
        _cap_results = json.load(f)
    print(f'Loaded {len(_cap_results)} captioning training results.')

if CAP_EVAL_FILE.exists():
    with open(CAP_EVAL_FILE) as f:
        _eval_results = json.load(f)
    _eval_by_tag = {r['tag']: r for r in _eval_results}
    print(f'Loaded evaluation results for {len(_eval_results)} variants.')
else:
    _eval_results = []
    _eval_by_tag  = {}
    print('ℹ  evaluation_results.json belum ada — jalankan captioning_evaluation.py')

# Vocab & references
_vocab = {}
_ref_map = {}
_test_feats = np.empty((0, 2048), dtype=np.float32)

if (PROC_DIR / 'vocab.json').exists():
    with open(PROC_DIR / 'vocab.json') as f:
        _vocab = json.load(f)
    print(f'Vocab size: {len(_vocab)}')
if (PROC_DIR / 'test_references.json').exists():
    with open(PROC_DIR / 'test_references.json') as f:
        _ref_map = json.load(f)
    print(f'Test references: {len(_ref_map)} images')
if (PROC_DIR / 'test_img_feats.npy').exists():
    _test_feats = np.load(PROC_DIR / 'test_img_feats.npy')
    print(f'Test features: {_test_feats.shape}')

_test_imgs_order = sorted(_ref_map.keys())

#### B.1. Perbandingan Jumlah Layer: RNN

Membandingkan RNN dengan **1**, **2**, **3** layer rekuren.

In [ ]:
# ── B.1 RNN layer count ────────────────────────────────────────────────────────
if not _cap_results:
    print('⚠  Skip.')
    _b1_results = []
else:
    _b1_results = sorted(
        [r for r in _cap_results if r['cell_type'] == 'rnn'],
        key=lambda r: (r['n_layers'], r['units']),
    )
    print('=' * 70)
    print('B.1  PERBANDINGAN JUMLAH LAYER: RNN')
    print('=' * 70)
    print_table(
        ['Tag', 'n_layers', 'units', 'BLEU-4', 'METEOR', 'time(s)', 'n_params'],
        [(r['tag'], r['n_layers'], r['units'],
          f"{r.get('test_bleu4', 0):.4f}",
          f"{_eval_by_tag.get(r['tag'], {}).get('meteor', 0):.4f}",
          f"{r['train_time']:.0f}",
          f"{r['n_params']:,}") for r in _b1_results]
    )

In [ ]:
# ── B.1 Plots ─────────────────────────────────────────────────────────────────
if not _b1_results:
    print('⚠  Skip.')
else:
    _b1_by_layer = defaultdict(list)
    for r in _b1_results:
        _b1_by_layer[r['n_layers']].append((r.get('test_bleu4', 0),
                                            _eval_by_tag.get(r['tag'], {}).get('meteor', 0)))
    _b1_layers  = sorted(_b1_by_layer.keys())
    _b1_bleu    = [np.mean([x[0] for x in _b1_by_layer[l]]) for l in _b1_layers]
    _b1_meteor  = [np.mean([x[1] for x in _b1_by_layer[l]]) for l in _b1_layers]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    compare_bar(_b1_bleu,
                [f'{l} layer{"s" if l>1 else ""}' for l in _b1_layers],
                title='RNN: BLEU-4 per Jumlah Layer (avg over hidden sizes)',
                ylabel='BLEU-4', ax=axes[0])
    compare_bar(_b1_meteor,
                [f'{l} layer{"s" if l>1 else ""}' for l in _b1_layers],
                title='RNN: METEOR per Jumlah Layer (avg)',
                ylabel='METEOR', ax=axes[1], color='mediumseagreen')
    plt.suptitle('B.1 RNN — Pengaruh Jumlah Layer', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    plot_loss_grid(_b1_results, n_cols=3)

#### [Kesimpulan B.1]

> *Isi setelah eksperimen selesai dijalankan.*

#### B.2. Perbandingan Jumlah Layer: LSTM

Membandingkan LSTM dengan **1**, **2**, **3** layer rekuren.

In [ ]:
# ── B.2 LSTM layer count ───────────────────────────────────────────────────────
if not _cap_results:
    print('⚠  Skip.')
    _b2_results = []
else:
    _b2_results = sorted(
        [r for r in _cap_results if r['cell_type'] == 'lstm'],
        key=lambda r: (r['n_layers'], r['units']),
    )
    print('=' * 70)
    print('B.2  PERBANDINGAN JUMLAH LAYER: LSTM')
    print('=' * 70)
    print_table(
        ['Tag', 'n_layers', 'units', 'BLEU-4', 'METEOR', 'time(s)', 'n_params'],
        [(r['tag'], r['n_layers'], r['units'],
          f"{r.get('test_bleu4', 0):.4f}",
          f"{_eval_by_tag.get(r['tag'], {}).get('meteor', 0):.4f}",
          f"{r['train_time']:.0f}",
          f"{r['n_params']:,}") for r in _b2_results]
    )

In [ ]:
# ── B.2 Plots ─────────────────────────────────────────────────────────────────
if not _b2_results:
    print('⚠  Skip.')
else:
    _b2_by_layer = defaultdict(list)
    for r in _b2_results:
        _b2_by_layer[r['n_layers']].append((r.get('test_bleu4', 0),
                                            _eval_by_tag.get(r['tag'], {}).get('meteor', 0)))
    _b2_layers = sorted(_b2_by_layer.keys())
    _b2_bleu   = [np.mean([x[0] for x in _b2_by_layer[l]]) for l in _b2_layers]
    _b2_meteor = [np.mean([x[1] for x in _b2_by_layer[l]]) for l in _b2_layers]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    compare_bar(_b2_bleu,
                [f'{l} layer{"s" if l>1 else ""}' for l in _b2_layers],
                title='LSTM: BLEU-4 per Jumlah Layer (avg over hidden sizes)',
                ylabel='BLEU-4', ax=axes[0])
    compare_bar(_b2_meteor,
                [f'{l} layer{"s" if l>1 else ""}' for l in _b2_layers],
                title='LSTM: METEOR per Jumlah Layer (avg)',
                ylabel='METEOR', ax=axes[1], color='darkorange')
    plt.suptitle('B.2 LSTM — Pengaruh Jumlah Layer', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    plot_loss_grid(_b2_results, n_cols=3)

#### [Kesimpulan B.2]

> *Isi setelah eksperimen selesai dijalankan.*

#### B.3. Perbandingan Ukuran Hidden State: RNN

Membandingkan RNN dengan hidden size **128** vs **512**.

In [ ]:
# ── B.3 RNN hidden size ────────────────────────────────────────────────────────
if not _cap_results:
    print('⚠  Skip.')
    _b3_results = []
else:
    _b3_results = sorted(
        [r for r in _cap_results if r['cell_type'] == 'rnn'],
        key=lambda r: (r['units'], r['n_layers']),
    )
    print('=' * 70)
    print('B.3  PERBANDINGAN UKURAN HIDDEN STATE: RNN')
    print('=' * 70)
    print_table(
        ['Tag', 'n_layers', 'units', 'BLEU-4', 'METEOR', 'n_params'],
        [(r['tag'], r['n_layers'], r['units'],
          f"{r.get('test_bleu4', 0):.4f}",
          f"{_eval_by_tag.get(r['tag'], {}).get('meteor', 0):.4f}",
          f"{r['n_params']:,}") for r in _b3_results]
    )

In [ ]:
# ── B.3 Plots ─────────────────────────────────────────────────────────────────
if not _b3_results:
    print('⚠  Skip.')
else:
    _b3_by_units = defaultdict(list)
    for r in _b3_results:
        _b3_by_units[r['units']].append((r.get('test_bleu4', 0),
                                         _eval_by_tag.get(r['tag'], {}).get('scratch_time',
                                                                             r.get('train_time', 0))))
    _b3_units     = sorted(_b3_by_units.keys())
    _b3_bleu_mean = [np.mean([x[0] for x in _b3_by_units[u]]) for u in _b3_units]
    _b3_time_mean = [np.mean([x[1] for x in _b3_by_units[u]]) for u in _b3_units]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    compare_bar(_b3_bleu_mean,
                [f'units={u}' for u in _b3_units],
                title='RNN: BLEU-4 per Hidden Size (avg over layers)',
                ylabel='BLEU-4', ax=axes[0])
    bars = axes[1].bar([f'units={u}' for u in _b3_units], _b3_time_mean,
                       color='darkorange', edgecolor='white')
    for bar, v in zip(bars, _b3_time_mean):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                     f'{v:.1f}s', ha='center', va='bottom', fontsize=10)
    axes[1].set_title('RNN: Waktu per Hidden Size (avg)', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('Waktu (s)', fontsize=12)
    axes[1].grid(True, alpha=0.3, axis='y')
    plt.suptitle('B.3 RNN — Pengaruh Hidden Size', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    plot_loss_grid(_b3_results, n_cols=3)

#### [Kesimpulan B.3]

> *Isi setelah eksperimen selesai dijalankan.*

#### B.4. Perbandingan Ukuran Hidden State: LSTM

Membandingkan LSTM dengan hidden size **128** vs **512**.

In [ ]:
# ── B.4 LSTM hidden size ───────────────────────────────────────────────────────
if not _cap_results:
    print('⚠  Skip.')
    _b4_results = []
else:
    _b4_results = sorted(
        [r for r in _cap_results if r['cell_type'] == 'lstm'],
        key=lambda r: (r['units'], r['n_layers']),
    )
    print('=' * 70)
    print('B.4  PERBANDINGAN UKURAN HIDDEN STATE: LSTM')
    print('=' * 70)
    print_table(
        ['Tag', 'n_layers', 'units', 'BLEU-4', 'METEOR', 'n_params'],
        [(r['tag'], r['n_layers'], r['units'],
          f"{r.get('test_bleu4', 0):.4f}",
          f"{_eval_by_tag.get(r['tag'], {}).get('meteor', 0):.4f}",
          f"{r['n_params']:,}") for r in _b4_results]
    )

In [ ]:
# ── B.4 Plots ─────────────────────────────────────────────────────────────────
if not _b4_results:
    print('⚠  Skip.')
else:
    _b4_by_units = defaultdict(list)
    for r in _b4_results:
        _b4_by_units[r['units']].append((r.get('test_bleu4', 0),
                                         _eval_by_tag.get(r['tag'], {}).get('scratch_time',
                                                                             r.get('train_time', 0))))
    _b4_units     = sorted(_b4_by_units.keys())
    _b4_bleu_mean = [np.mean([x[0] for x in _b4_by_units[u]]) for u in _b4_units]
    _b4_time_mean = [np.mean([x[1] for x in _b4_by_units[u]]) for u in _b4_units]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    compare_bar(_b4_bleu_mean,
                [f'units={u}' for u in _b4_units],
                title='LSTM: BLEU-4 per Hidden Size (avg over layers)',
                ylabel='BLEU-4', ax=axes[0])
    bars = axes[1].bar([f'units={u}' for u in _b4_units], _b4_time_mean,
                       color='mediumseagreen', edgecolor='white')
    for bar, v in zip(bars, _b4_time_mean):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                     f'{v:.1f}s', ha='center', va='bottom', fontsize=10)
    axes[1].set_title('LSTM: Waktu per Hidden Size (avg)', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('Waktu (s)', fontsize=12)
    axes[1].grid(True, alpha=0.3, axis='y')
    plt.suptitle('B.4 LSTM — Pengaruh Hidden Size', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    plot_loss_grid(_b4_results, n_cols=3)

#### [Kesimpulan B.4]

> *Isi setelah eksperimen selesai dijalankan.*

#### B.5. Perbandingan RNN vs LSTM

Membandingkan arsitektur terbaik RNN dengan arsitektur terbaik LSTM secara
kuantitatif (BLEU-4, METEOR) dan kualitatif (10 gambar test).

In [ ]:
# ── B.5 Best RNN vs best LSTM: quantitative ───────────────────────────────────
if not _cap_results:
    print('⚠  Skip.')
    _best_rnn = _best_lstm = None
else:
    _rnn_pool  = [r for r in _cap_results if r['cell_type'] == 'rnn']
    _lstm_pool = [r for r in _cap_results if r['cell_type'] == 'lstm']
    _best_rnn  = max(_rnn_pool,  key=lambda r: r.get('test_bleu4', 0)) if _rnn_pool  else None
    _best_lstm = max(_lstm_pool, key=lambda r: r.get('test_bleu4', 0)) if _lstm_pool else None

if _best_rnn and _best_lstm:
    _rnn_eval  = _eval_by_tag.get(_best_rnn['tag'],  {})
    _lstm_eval = _eval_by_tag.get(_best_lstm['tag'], {})

    print('=' * 70)
    print('B.5  PERBANDINGAN RNN VS LSTM')
    print('=' * 70)
    print_table(
        ['Model', 'Tag', 'n_layers', 'units', 'BLEU-4', 'METEOR', 'time(s)', 'n_params'],
        [
            ('RNN',  _best_rnn['tag'],  _best_rnn['n_layers'],  _best_rnn['units'],
             f"{_best_rnn.get('test_bleu4', 0):.4f}",
             f"{_rnn_eval.get('meteor', 0):.4f}",
             f"{_rnn_eval.get('scratch_time', _best_rnn.get('train_time', 0)):.0f}",
             f"{_best_rnn['n_params']:,}"),
            ('LSTM', _best_lstm['tag'], _best_lstm['n_layers'], _best_lstm['units'],
             f"{_best_lstm.get('test_bleu4', 0):.4f}",
             f"{_lstm_eval.get('meteor', 0):.4f}",
             f"{_lstm_eval.get('scratch_time', _best_lstm.get('train_time', 0)):.0f}",
             f"{_best_lstm['n_params']:,}"),
        ]
    )

In [ ]:
# ── B.5 Loss curves side-by-side ──────────────────────────────────────────────
if not _best_rnn or not _best_lstm:
    print('⚠  Skip.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, r, color, label in [
        (axes[0], _best_rnn,  'steelblue',  'RNN'),
        (axes[1], _best_lstm, 'darkorange', 'LSTM'),
    ]:
        ep = range(1, len(r['train_loss']) + 1)
        ax.plot(ep, r['train_loss'], color=color, label='train')
        ax.plot(ep, r['val_loss'],   color=color, linestyle='--', alpha=0.8, label='val')
        ax.set_title(f'{label}: {r["tag"]}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=11)
        ax.set_ylabel('Loss', fontsize=11)
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.suptitle('B.5 Loss Curves — Best RNN vs Best LSTM',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── B.5 Qualitative: decode with ScratchDecoder ───────────────────────────────
_qual_ok = (_HAS_TF and _best_rnn is not None and _best_lstm is not None
            and _vocab and len(_test_feats) > 0)

if not _qual_ok:
    print('⚠  Skip qualitative — TF/vocab/features/models belum tersedia.')
else:
    from captioning_evaluation import ScratchDecoder, extract_weights

    _rnn_keras  = keras.models.load_model(_best_rnn['saved_to'])
    _lstm_keras = keras.models.load_model(_best_lstm['saved_to'])
    _rnn_w      = extract_weights(_rnn_keras,  'rnn',  _best_rnn['n_layers'])
    _lstm_w     = extract_weights(_lstm_keras, 'lstm', _best_lstm['n_layers'])
    _rnn_dec    = ScratchDecoder('rnn',  _rnn_w,  _vocab, EMBED_DIM,
                                 [_best_rnn['units']]  * _best_rnn['n_layers'])
    _lstm_dec   = ScratchDecoder('lstm', _lstm_w, _vocab, EMBED_DIM,
                                 [_best_lstm['units']] * _best_lstm['n_layers'])

    _N_QUAL = min(50, len(_test_feats))
    print(f'Generating captions for {_N_QUAL} images ...')
    _rnn_caps  = [_rnn_dec.generate(_test_feats[i])  for i in range(_N_QUAL)]
    _lstm_caps = [_lstm_dec.generate(_test_feats[i]) for i in range(_N_QUAL)]
    print('Done.')

In [ ]:
# ── B.5 Qualitative: select high/mid/low scoring images ──────────────────────
if not _qual_ok:
    print('⚠  Skip.')
else:
    smooth = SmoothingFunction().method1

    _per_img = []
    for i in range(_N_QUAL):
        img_name = _test_imgs_order[i] if i < len(_test_imgs_order) else ''
        refs     = [r.split() for r in _ref_map.get(img_name, ['a'])]
        b_rnn    = sentence_bleu(refs, _rnn_caps[i],  smoothing_function=smooth)
        b_lstm   = sentence_bleu(refs, _lstm_caps[i], smoothing_function=smooth)
        _per_img.append({'idx': i, 'img_name': img_name,
                         'b_rnn': b_rnn, 'b_lstm': b_lstm, 'b_avg': (b_rnn+b_lstm)/2})

    _per_img.sort(key=lambda x: -x['b_avg'])
    n_show = 10
    step   = max(1, len(_per_img) // n_show)
    hi     = _per_img[:n_show//3]
    lo     = _per_img[-(n_show//3):]
    mid_start = len(_per_img)//2 - (n_show - len(hi) - len(lo))//2
    mid    = _per_img[mid_start: mid_start + (n_show - len(hi) - len(lo))]
    _show_items = hi + mid + lo
    _show_items = _show_items[:n_show]
    print(f'Selected {len(_show_items)} images: {n_show//3} high, {len(mid)} mid, {n_show//3} low BLEU-4')

In [ ]:
# ── B.5 Qualitative: display image grid ───────────────────────────────────────
if not _qual_ok or not _show_items:
    print('⚠  Skip.')
else:
    n_rows = len(_show_items)
    fig, axes = plt.subplots(n_rows, 2, figsize=(16, n_rows * 3.5))
    if n_rows == 1:
        axes = np.array([axes])

    for row, item in enumerate(_show_items):
        idx      = item['idx']
        img_name = item['img_name']
        img_path = FLICKR_IMG_DIR / img_name
        refs     = _ref_map.get(img_name, ['(no reference)'])
        ref_str  = refs[0][:70] + ('...' if len(refs[0]) > 70 else '')

        for col in range(2):
            ax = axes[row, col]
            if img_path.exists():
                img = Image.open(img_path).convert('RGB')
                ax.imshow(img)
            else:
                ax.set_facecolor('#eeeeee')
                ax.text(0.5, 0.5, img_name[:25], ha='center', va='center',
                        transform=ax.transAxes, fontsize=8, color='gray')
            ax.axis('off')

        rnn_str  = ' '.join(_rnn_caps[idx])[:65]
        lstm_str = ' '.join(_lstm_caps[idx])[:65]
        tag_rnn  = 'HIGH' if item in hi else ('LOW' if item in lo else 'MID')

        axes[row, 0].set_title(
            f'[{row+1}/{tag_rnn}] RNN  BLEU={item["b_rnn"]:.3f}\n'
            f'Cap : {rnn_str}\nREF : {ref_str}',
            fontsize=7.5, loc='left', pad=3)
        axes[row, 1].set_title(
            f'[{row+1}/{tag_rnn}] LSTM BLEU={item["b_lstm"]:.3f}\n'
            f'Cap : {lstm_str}\nREF : {ref_str}',
            fontsize=7.5, loc='left', pad=3)

    plt.suptitle(
        f'B.5 Qualitative: RNN ({_best_rnn["tag"]}) vs LSTM ({_best_lstm["tag"]})\n'
        'Urutan: score tinggi → sedang → rendah',
        fontsize=13, fontweight='bold',
    )
    plt.tight_layout()
    plt.show()

#### [Kesimpulan B.5]

> *Isi setelah eksperimen selesai dijalankan.*
>
> **Kaitkan dengan:**
> - **Vanishing gradient**: RNN rentan terhadap menghilangnya gradien saat backprop
>   melewati banyak timestep, sehingga informasi dari token jauh sulit dipelajari.
>   LSTM memitigasi ini via *forget gate*, *input gate*, dan *cell state* yang menjaga
>   gradien mengalir lebih stabil.
> - **Memori jangka panjang**: Cell state LSTM bertindak sebagai "jalur cepat" yang
>   membawa konteks antar langkah waktu tanpa transformasi berulang, memungkinkan
>   LSTM mengingat kata-kata dari awal kalimat saat memprediksi kata di akhir caption.
> - **Implikasi empiris**: lihat perbedaan BLEU-4 dan contoh caption di atas —
>   apakah LSTM menghasilkan caption yang lebih koheren secara gramatikal?

#### B.6. Perbandingan Keras vs From Scratch (Captioning)

Menggunakan arsitektur terbaik keseluruhan: BLEU-4 Keras (teacher forcing)
dibandingkan dengan BLEU-4 from-scratch (greedy decoding).

In [ ]:
# ── B.6 Keras vs From Scratch: captioning ─────────────────────────────────────
if not _cap_results or not _eval_results:
    print('⚠  Skip — captioning_evaluation.py belum dijalankan.')
    _b6_ok = False
else:
    _b6_best = max(_eval_results, key=lambda r: r.get('bleu4', 0))
    _b6_tag  = _b6_best['tag']
    _b6_cap_row = next((r for r in _cap_results if r['tag'] == _b6_tag), None)
    _b6_ok   = True

    print('=' * 70)
    print('B.6  PERBANDINGAN KERAS VS FROM SCRATCH (CAPTIONING)')
    print('=' * 70)
    print(f'  Arsitektur terbaik       : {_b6_tag}')
    if _b6_cap_row:
        print(f'  Jumlah parameter         : {_b6_cap_row["n_params"]:,}')
    print()
    print(f'  Keras BLEU-4             : {_b6_best.get("keras_bleu4", 0):.4f}')
    print(f'  Scratch BLEU-4           : {_b6_best.get("bleu4", 0):.4f}')
    print(f'  Scratch METEOR           : {_b6_best.get("meteor", 0):.4f}')
    print(f'  |diff| BLEU-4            : {abs(_b6_best.get("keras_bleu4",0)-_b6_best.get("bleu4",0)):.6f}')
    print(f'  Waktu decode (scratch)   : {_b6_best.get("scratch_time", 0):.1f}s')

In [ ]:
# ── B.6 Plots ─────────────────────────────────────────────────────────────────
if not _b6_ok:
    print('⚠  Skip.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    _b6_lbls  = ['Keras\n(teacher forcing)', 'From Scratch\n(greedy)']
    _b6_bleus = [_b6_best.get('keras_bleu4', 0), _b6_best.get('bleu4', 0)]
    _b6_mets  = [0.0, _b6_best.get('meteor', 0)]  # Keras METEOR not precomputed

    compare_bar(_b6_bleus, _b6_lbls,
                title=f'BLEU-4: Keras vs From Scratch\n({_b6_tag})',
                ylabel='BLEU-4', ax=axes[0],
                color=['steelblue', 'darkorange'], highlight_best=False)

    bars = axes[1].bar(_b6_lbls, _b6_mets, color=['steelblue', 'darkorange'], edgecolor='white')
    for bar, v in zip(bars, _b6_mets):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                     f'{v:.4f}' if v > 0 else 'N/A',
                     ha='center', va='bottom', fontsize=10)
    axes[1].set_title(f'METEOR: Keras vs From Scratch\n({_b6_tag})',
                      fontsize=12, fontweight='bold')
    axes[1].set_ylabel('METEOR', fontsize=12)
    axes[1].grid(True, alpha=0.3, axis='y')
    axes[1].set_ylim(0, max(_b6_mets) * 1.2 + 0.02)

    plt.suptitle('B.6 Keras vs From Scratch — Image Captioning',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    if _b6_cap_row:
        fig2, ax2 = plt.subplots(figsize=(10, 4))
        plot_loss_curves(
            [{'train_loss': _b6_cap_row['train_loss'], 'val_loss': _b6_cap_row['val_loss']}],
            [_b6_tag],
            title=f'B.6 Loss Curve — {_b6_tag}', ax=ax2,
        )
        plt.tight_layout()
        plt.show()

#### [Kesimpulan B.6]

> *Isi setelah eksperimen selesai dijalankan.*

#### B.7. Pengaruh Panjang Maksimum Caption terhadap BLEU-4

Menggunakan arsitektur terbaik keseluruhan, mengevaluasi BLEU-4 dan METEOR
pada variasi panjang maksimum caption: **15**, **25**, **35**, **50** token.

In [ ]:
# ── B.7 Max caption length sweep ──────────────────────────────────────────────
_b7_ok = (_HAS_TF and _HAS_NLTK and _vocab and len(_test_feats) > 0
          and bool(_cap_results))

if not _b7_ok:
    print('⚠  Skip — TF/NLTK/vocab/features/models belum tersedia.')
else:
    from captioning_evaluation import ScratchDecoder, extract_weights, evaluate_corpus

    _b7_best = max(_cap_results, key=lambda r: r.get('test_bleu4', 0))
    _b7_tag  = _b7_best['tag']
    print(f'Best model for B.7: {_b7_tag}')

    # Reuse decoder if B.5 used the same model
    try:
        if (_best_rnn and _b7_tag == _best_rnn['tag']):
            _dec_b7 = _rnn_dec
        elif (_best_lstm and _b7_tag == _best_lstm['tag']):
            _dec_b7 = _lstm_dec
        else:
            raise NameError()
        print('(Reusing B.5 decoder)')
    except NameError:
        _km_b7  = keras.models.load_model(_b7_best['saved_to'])
        _w_b7   = extract_weights(_km_b7, _b7_best['cell_type'], _b7_best['n_layers'])
        _dec_b7 = ScratchDecoder(_b7_best['cell_type'], _w_b7, _vocab, EMBED_DIM,
                                 [_b7_best['units']] * _b7_best['n_layers'])

    _max_lens  = [15, 25, 35, 50]
    _b7_rows   = []

    for ml in _max_lens:
        t0   = time.time()
        hyps = [_dec_b7.generate(_test_feats[i], ml) for i in range(len(_test_feats))]
        dt   = time.time() - t0
        m    = evaluate_corpus(hyps, _ref_map, _test_imgs_order)
        _b7_rows.append({'max_len': ml, 'bleu4': m['bleu4'], 'meteor': m['meteor'], 'time': dt})
        print(f'  max_len={ml:2d}  BLEU-4={m["bleu4"]:.4f}  METEOR={m["meteor"]:.4f}  ({dt:.1f}s)')

    print()
    print('=' * 70)
    print('B.7  PENGARUH PANJANG MAKSIMUM CAPTION')
    print('=' * 70)
    print(f'  Model: {_b7_tag}')
    print_table(
        ['max_len', 'BLEU-4', 'METEOR', 'time(s)'],
        [(r['max_len'], f"{r['bleu4']:.4f}", f"{r['meteor']:.4f}", f"{r['time']:.1f}")
         for r in _b7_rows]
    )

In [ ]:
# ── B.7 Plots ─────────────────────────────────────────────────────────────────
if '_b7_rows' not in dir() or not _b7_rows:
    print('⚠  Skip.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    _b7_lens  = [r['max_len'] for r in _b7_rows]
    _b7_bleus = [r['bleu4']   for r in _b7_rows]
    _b7_mets  = [r['meteor']  for r in _b7_rows]

    compare_bar(_b7_bleus, [str(l) for l in _b7_lens],
                title=f'BLEU-4 per max_len\n({_b7_tag})',
                ylabel='BLEU-4', ax=axes[0])

    compare_bar(_b7_mets, [str(l) for l in _b7_lens],
                title=f'METEOR per max_len\n({_b7_tag})',
                ylabel='METEOR', ax=axes[1], color='mediumseagreen')

    plt.suptitle('B.7 Pengaruh Panjang Maksimum Caption terhadap BLEU-4',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

#### [Kesimpulan B.7]

> *Isi setelah eksperimen selesai dijalankan.*